## BDE STATUS WISE REPORT

In [8]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from gspread_pandas import Spread, conf
import gspread
import warnings



warnings.filterwarnings('ignore', category=DeprecationWarning)

In [9]:
# --- 1. CONFIGURATION & CONNECTION ---
config_path = r'C:\Users\USER\Downloads\Emarath_global\service_account.json'
emarath_global_sheet_url = "https://docs.google.com/spreadsheets/d/1aXobnCl4QQ-hQ_IWoTPo2ZEo3HM5kNwycwMBJtEDOfc/edit?gid=822921427#gid=822921427"
target_sheet_url = "https://docs.google.com/spreadsheets/d/19B1Vclt9U0AOu-yk99E960g5tYRQ9JRX-M2GlAPvE6U/edit?pli=1&gid=1513925117#gid=1513925117"

c = conf.get_config(file_name=config_path)
spread = Spread(emarath_global_sheet_url, config=c)
target_spread = Spread(target_sheet_url, config=c)

In [10]:

def safe_upload(spreadsheet, df, sheet_name):
    if not df.empty:
        spreadsheet.open_sheet(sheet_name, create=True)
        spreadsheet.sheet.batch_clear(['A2:Z10000']) 
        upload_df = df.astype(str)
        spreadsheet.df_to_sheet(upload_df, index=False, replace=False, headers=False, start="A2")


In [16]:

target_bde_sheets = [
    'RAHIB', 'HABIYA', 'BURHANA', 'SHAMNA', 'ARUN', 
    'CHAITHANYA', 'ZAKIYA', 'SAFAN', 'SUSHANTHIKA', 
    'ADWAITHA', 'NEHA', 'GOWTHAM', 'AMINA', 'NAFI',
    'RINSIYA', 'ARSHAD',
    # 'SHABNA', 'FARSANA', 'AJIN', 'ADHIL', 
]

lead_statuses = ["WON", "SUPER HOT", "HOT", "WARM", "COLD", "BOOKING", "WHATS APP ENGAGE"]

c = conf.get_config(file_name=config_path)
emarath_spread = Spread(emarath_global_sheet_url, config=c)


all_data_list = []

for sheet_name in target_bde_sheets:
    try:
        df = emarath_spread.sheet_to_df(sheet=sheet_name, index=None, header_rows=1)
        
        mapping_cols = {
            'REF NO'      : 'REF NO',
            'COUNTRY'     : 'REGION',
            'DATE'        : 'DATE',
            'AGENT'       : 'AGENT',
            'CUSTOMER PATH': 'CUSTOMER PATH',
            'NAME'        : 'NAME',
            'PHONE NO 1'  : 'PHONE NO',
            'STATUS'      : 'STATUS',
            'PRODUCT 1'   : 'PRODUCT',
            'CALL STATUS' : 'CALL STATUS'
        }
        

        if not df.empty and all(col in df.columns for col in mapping_cols.keys()):
            subset = df[list(mapping_cols.keys())].copy()
            subset.rename(columns=mapping_cols, inplace=True)

            subset['DATE'] = pd.to_datetime(subset['DATE'], errors='coerce', dayfirst=True)

            subset = subset[
                ~subset['PHONE NO'].astype(str).str.strip().str.upper().isin(['', 'NAN', 'NONE', 'N/A', 'UNKNOWN', '0'])
            ]

            if subset.empty:
                continue

            for col in ['PHONE NO', 'CUSTOMER PATH', 'CALL STATUS', 'STATUS']:
                subset[col] = subset[col].astype(str).str.strip().str.upper()

            all_data_list.append(subset)
                
    except Exception as e:
        print(f"Error in sheet {sheet_name}: {e}")

if all_data_list:
    master_df = pd.concat(all_data_list, ignore_index=True)
    
    unattended_mask = (master_df['CALL STATUS'] == '') | (master_df['CALL STATUS'] == 'NAN')
    unattended_df = master_df[unattended_mask].copy()
    
    if not unattended_df.empty:

        unattended_df = unattended_df.sort_values(by=['AGENT', 'DATE'], ascending=[True, True])
        safe_upload(target_spread, unattended_df, "UNATTENDED_LEADS")
        print(f"Uploaded {len(unattended_df)} unattended leads.")


    attended_df = master_df[~unattended_mask].copy()

    for status in lead_statuses:
        status_df = attended_df[attended_df['STATUS'] == status.upper()].copy()

        if not status_df.empty:
            status_df = status_df.sort_values(by=['DATE'], ascending=[True])
            
            tab_name = status.replace(' ', '_')[:31]
            safe_upload(target_spread, status_df, tab_name)
            print(f"Uploaded {len(status_df)} leads to {tab_name}")
        

Uploaded 787 unattended leads.
Uploaded 665 leads to WON
Uploaded 106 leads to SUPER_HOT
Uploaded 139 leads to HOT
Uploaded 285 leads to WARM
Uploaded 2218 leads to COLD
Uploaded 52 leads to BOOKING
Uploaded 979 leads to WHATS_APP_ENGAGE
